In [0]:
from pyspark.sql import functions as F

source_folders = [
    "s3://project-atlas-olist-20261024-180312/raw/olist/orders/",
    "s3://project-atlas-olist-20261024-180312/raw/olist/order_items/",
    "s3://project-atlas-olist-20261024-180312/raw/olist/products/",
]

manifest_table = "workspace.atlas_bronze.file_manifest"

print("Discovery configuration loaded.")

Discovery configuration loaded.


In [0]:
discovered_files = []

for folder in source_folders:
    for file_info in dbutils.fs.ls(folder):
        if file_info.name.lower().endswith(".csv"):
            discovered_files.append({
                "source_path": file_info.path,
                "file_size_bytes": int(file_info.size),
                "last_modified_ms": int(file_info.modificationTime)
            })

print(f"CSV files discovered in S3: {len(discovered_files)}")

if not discovered_files:
    raise ValueError(
        "No CSV files found. Check S3 access and configured folders."
    )

discovered_df = (
    spark.createDataFrame(discovered_files)
    .withColumn(
        "current_modified",
        F.expr("timestamp_millis(last_modified_ms)")
    )
)

display(discovered_df)

Files discovered: 4


source_path,file_status,file_size_bytes,discovery_status
s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,PROCESSED,15438671,UNCHANGED
s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,PROCESSED,17654914,UNCHANGED
s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,PROCESSED,2379446,UNCHANGED
s3://project-atlas-olist-20261024-180312/test-discovery/discovery_test.csv,null,33,NEW


Files requiring processing: 1
Test files requiring processing: 1
has_pending_files = True
Files requiring processing: 1
has_pending_files = True


In [0]:
manifest_df = (
    spark.table(manifest_table)
    .select(
        "source_path",
        "file_status",
        F.col("file_size_bytes").alias("manifest_size"),
        F.col("last_modified").alias("manifest_modified")
    )
)

display(manifest_df)

TRUE BRANCH EXECUTED
Discovery successfully detected the temporary test file.
Production Bronze, Silver, and Gold tables were not modified.


In [0]:
comparison = discovered_df.join(
    manifest_df,
    on="source_path",
    how="left"
)

classified = comparison.withColumn(
    "discovery_status",
    F.when(
        F.col("file_status").isNull(),
        F.lit("NEW")
    ).when(
        F.col("manifest_size").isNull()
        | F.col("manifest_modified").isNull(),
        F.lit("METADATA_MISSING")
    ).when(
        F.col("file_size_bytes") != F.col("manifest_size"),
        F.lit("CHANGED")
    ).when(
        F.col("current_modified") != F.col("manifest_modified"),
        F.lit("CHANGED")
    ).otherwise(
        F.lit("UNCHANGED")
    )
)

display(
    classified.select(
        "source_path",
        "file_status",
        "file_size_bytes",
        "current_modified",
        "discovery_status"
    ).orderBy("source_path")
)

In [0]:
pending_statuses = [
    "NEW",
    "CHANGED",
    "METADATA_MISSING"
]

pending_files = classified.filter(
    F.col("discovery_status").isin(pending_statuses)
)

pending_count = pending_files.count()
has_pending_files = pending_count > 0

print(f"Files requiring processing: {pending_count}")
print(f"has_pending_files = {has_pending_files}")

display(pending_files.select(
    "source_path",
    "discovery_status",
    "file_size_bytes",
    "current_modified"
))

In [0]:

dbutils.jobs.taskValues.set(
    key="has_pending_files",
    value=has_pending_files
)

pending_paths = [
    row["source_path"]
    for row in pending_files.select("source_path").collect()
]

dbutils.jobs.taskValues.set(
    key="pending_file_paths",
    value=pending_paths
)

print(f"Files requiring processing: {len(pending_paths)}")
print(f"has_pending_files = {has_pending_files}")

for path in pending_paths:
    print(path)
